# ARM-Gym GRPO Training (Kaggle T4)

Train Qwen2.5-Coder-7B to generate optimized AArch64 assembly that beats gcc -O3.

**Accelerator**: GPU T4 x1 or T4 x2 (single GPU used)
**Internet**: ON (model download)
**Time**: ~6 hours for 100 steps

## 0. Environment setup (run first)

In [ ]:
import os

# Kaggle T4 x2 launches 2 distributed workers and sets these vars before the notebook runs.
# Unsetting them forces single-process mode in accelerate/transformers.
for _v in ["MASTER_ADDR", "MASTER_PORT", "RANK", "LOCAL_RANK", "WORLD_SIZE",
           "TORCHELASTIC_RESTART_COUNT", "GROUP_RANK", "ROLE_RANK", "ROLE_NAME",
           "TORCHELASTIC_USE_AGENT_STORE"]:
    os.environ.pop(_v, None)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("single GPU mode: distributed vars cleared, GPU 0 only")

## 1. Install ARM cross-compilation toolchain

In [ ]:
%%bash
set -e
echo "[1/2] apt toolchain"
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq   gcc-aarch64-linux-gnu binutils-aarch64-linux-gnu >/dev/null

echo "[2/2] llvm"
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang llvm llvm-dev >/dev/null
# Try llvm-21 from apt.llvm.org (best effort)
curl -fsSL https://apt.llvm.org/llvm-snapshot.gpg.key   | gpg --dearmor -o /usr/share/keyrings/llvm.gpg 2>/dev/null || true
CODENAME=$(lsb_release -cs)
echo "deb [signed-by=/usr/share/keyrings/llvm.gpg] http://apt.llvm.org/${CODENAME}/ llvm-toolchain-${CODENAME}-21 main"   > /etc/apt/sources.list.d/llvm21.list 2>/dev/null || true
apt-get update -qq 2>/dev/null || true
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang-21 llvm-21 >/dev/null 2>&1   && echo "llvm-21 installed" || echo "llvm-21 unavailable, using default"

echo "verify"
which aarch64-linux-gnu-gcc
which llvm-mca-21 2>/dev/null || which llvm-mca
echo "toolchain done"

## 2. Install training stack

In [ ]:
import subprocess, sys
cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "trl>=0.16", "peft", "accelerate",
     "datasets", "pydantic", "numpy", "matplotlib", "httpx"],
    [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"],  # kept for optional use
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"WARN: {' '.join(cmd[:5])}... failed")
        print(r.stderr[-300:])
    else:
        print(f"OK: {' '.join(cmd[:5])}...")
print("deps done")

## 3. Unpack arm_gym source

In [ ]:
import base64, io, os, tarfile

B64 = (
    "..."
    "5m/zt/nb/G3+Nn+bvzv+/n8p382lALgBAA=="
)

data = base64.b64decode(B64)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
    tar.extractall(".")
print("extracted arm_gym + kaggle source")
for root, dirs, files in os.walk("arm_gym"):
    for f in files:
        if f.endswith(".py"):
            print(f"  {os.path.join(root, f)}")

## 4. Smoke test

In [ ]:
import os, sys, shutil

# Add llvm bin dir to PATH
for d in ["/usr/lib/llvm-21/bin", "/usr/lib/llvm-20/bin", "/usr/lib/llvm-14/bin"]:
    if os.path.isdir(d):
        os.environ["PATH"] = d + ":" + os.environ["PATH"]
        print(f"LLVM path: {d}")
        break

sys.path.insert(0, os.getcwd())

from arm_gym.compile_baseline import detect_toolchain
tc = detect_toolchain()
print(f"clang={tc.clang} gcc={tc.gcc_aarch64} mca={tc.mca} mcpu={tc.mcpu}")
assert tc.ready(), "toolchain not ready - rerun cell 1"
assert tc.mca, "llvm-mca not found - rerun cell 1"

from arm_gym.kernels import summary, generate_variants
s = summary()
print(f"templates={s['templates']} variants={s['variants']}")

from arm_gym.compile_baseline import compile_to_asm
v = next(generate_variants("vec_add"))
asm = compile_to_asm(v.c_source, tc)
print(f"compiled vec_add, asm length={len(asm)}")

from arm_gym.mca import run_mca
rep = run_mca(asm, tc.mca, tc.mcpu)
print(f"MCA: cycles={rep.total_cycles} ipc={rep.ipc:.2f}")
print("smoke OK")

## 5. GPU detection

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU - enable T4 in notebook settings")

## 6. Build training dataset

In [ ]:
import os, sys

for d in ["/usr/lib/llvm-21/bin", "/usr/lib/llvm-20/bin", "/usr/lib/llvm-14/bin"]:
    if os.path.isdir(d):
        os.environ["PATH"] = d + ":" + os.environ["PATH"]
        break

sys.path.insert(0, os.getcwd())

from arm_gym.compile_baseline import detect_toolchain
from kaggle.dataset import DatasetConfig, build as build_dataset

tc = detect_toolchain()

cfg = DatasetConfig(max_train=128, max_eval=16, difficulty_max=1)
train_ds, eval_ds, lookup = build_dataset(tc, cfg, tokenizer=None)
print(f"train={len(train_ds)} eval={len(eval_ds)} lookup={len(lookup)}")
print(f"sample prompt length: {len(train_ds[0]['prompt'])} chars")

## 7. GRPO Training

PEFT QLoRA 4-bit, bfloat16, single T4.
- Model: Qwen2.5-Coder-7B-Instruct (~5GB VRAM 4-bit)
- lora_rank=48, lora_alpha=48, all 7 attention+MLP modules
- num_generations=8, temperature=0.9, DAPO loss
- 100 steps (~6 hours on T4 with 7B 4-bit)

In [ ]:
import os, sys, time, csv, re
import torch
from pathlib import Path

for d in ["/usr/lib/llvm-21/bin", "/usr/lib/llvm-20/bin", "/usr/lib/llvm-14/bin"]:
    if os.path.isdir(d):
        os.environ["PATH"] = d + ":" + os.environ["PATH"]
        break

sys.path.insert(0, os.getcwd())

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_RANK = 48
LORA_ALPHA = 48          # scaling=1.0; safer than 2.0 for 7B+DAPO
STEPS = 100
NUM_GENERATIONS = 8
MAX_PROMPT_LEN = 1024
MAX_COMPLETION_LEN = 512
LR = 5e-6
BATCH_SIZE = 1
GRAD_ACCUM = 4           # 4 prompts × 8 gens = 32 completions/step
OUT_DIR = "/kaggle/working/runs/grpo"

os.makedirs(OUT_DIR, exist_ok=True)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainerCallback

import gc
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info(0)
print(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
if free < 4e9:
    raise RuntimeError(
        f"Only {free/1e9:.1f} GB free. "
        "Restart the Kaggle kernel (Run > Restart Session) then run all cells from Cell 0."
    )

print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit (7B ~4GB VRAM) ...")
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)
lora = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
free2, _ = torch.cuda.mem_get_info(0)
print(f"Model loaded. GPU free: {free2/1e9:.1f} GB")

from arm_gym.compile_baseline import detect_toolchain
from kaggle.dataset import DatasetConfig, build as build_dataset

tc = detect_toolchain()
ds_cfg = DatasetConfig(max_train=256, max_eval=32, difficulty_max=2)
train_ds, eval_ds, _ = build_dataset(tc, ds_cfg, tokenizer=tokenizer)
print(f"dataset: train={len(train_ds)} eval={len(eval_ds)}")

# ── step logger callback ──────────────────────────────────────────────────────

class StepLogger(TrainerCallback):
    def __init__(self, total, csv_path):
        self.total = total
        self.start = time.time()
        self.csv_path = csv_path
        self._f = None
        self._writer = None
        self._fields = []

    def _fmt(self, v):
        if v is None or v == "":
            return "-"
        try:
            return f"{float(v):.4f}"
        except (TypeError, ValueError):
            return str(v)

    def _vram(self):
        try:
            used = torch.cuda.memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            return f"{used:.1f}/{total:.1f}GB"
        except Exception:
            return "n/a"

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        elapsed = time.time() - self.start
        rate = step / elapsed if elapsed > 0 and step > 0 else 0
        eta = (self.total - step) / rate if rate > 0 else 0
        pct = step / self.total * 100
        print(
            f"[step {step:>3}/{self.total}] {pct:5.1f}%  "
            f"elapsed={elapsed/60:5.1f}m  eta={eta/60:5.1f}m  "
            f"loss={self._fmt(logs.get('loss'))}  "
            f"reward={self._fmt(logs.get('reward'))}  "
            f"correct={self._fmt(logs.get('rewards/correctness_reward/mean'))}  "
            f"speedup={self._fmt(logs.get('rewards/speedup_reward/mean'))}  "
            f"vram={self._vram()}",
            flush=True,
        )
        row = {"step": step, "elapsed_s": f"{elapsed:.1f}", **logs}
        new_keys = [k for k in row if k not in self._fields]
        if new_keys:
            self._fields.extend(new_keys)
            reopen = self._f is not None
            if reopen:
                self._f.close()
            self._f = open(self.csv_path, "a" if reopen else "w", newline="", buffering=1)
            self._writer = csv.DictWriter(self._f, fieldnames=self._fields, extrasaction="ignore")
            if not reopen:
                self._writer.writeheader()
        if self._writer:
            self._writer.writerow({k: row.get(k, "") for k in self._fields})
            self._f.flush()

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step % 20 == 0 and step > 0:
            gc.collect()
            torch.cuda.empty_cache()
            try:
                from kaggle import reward_fn as _rf
                cache = getattr(_rf, "_CACHE", {})
                if len(cache) > 5000:
                    keys = list(cache.keys())
                    for k in keys[:-2000]:
                        cache.pop(k, None)
                print(f"  [mem] step={step} vram={self._vram()} cache={len(cache)}", flush=True)
            except Exception:
                pass

    def on_train_end(self, args, state, control, **kwargs):
        if self._f:
            self._f.close()
        print(f"\n[done] training complete at step {state.global_step}", flush=True)

# ── GRPO config ───────────────────────────────────────────────────────────────

from kaggle.reward_fn import syntax_reward, correctness_reward, speedup_reward
from trl import GRPOConfig, GRPOTrainer

grpo_params = dict(
    output_dir=OUT_DIR,
    max_steps=STEPS,
    learning_rate=LR,
    warmup_steps=20,
    lr_scheduler_type="constant_with_warmup",
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    generation_batch_size=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    gradient_checkpointing=True,
    bf16=True,
    max_grad_norm=0.1,
    temperature=0.9,
    loss_type="dapo",
    beta=0.0,
    epsilon=0.2,
    epsilon_high=0.28,
    mask_truncated_completions=True,
    multi_objective_aggregation="normalize_then_sum",
    remove_unused_columns=False,
    logging_steps=1,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
)

while True:
    try:
        gcfg = GRPOConfig(**grpo_params)
        break
    except TypeError as exc:
        m = re.search(r"unexpected keyword argument '(\w+)'", str(exc))
        if not m:
            raise
        dropped = m.group(1)
        print(f"TRL compat: dropping {dropped!r}")
        grpo_params.pop(dropped, None)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[syntax_reward, correctness_reward, speedup_reward],
    args=gcfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    callbacks=[StepLogger(STEPS, f"{OUT_DIR}/log.csv")],
)

try:
    trainer.train()
except KeyboardInterrupt:
    print("\n[interrupted] saving checkpoint...", flush=True)
    trainer.save_model(f"{OUT_DIR}/interrupted")

trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")
print(f"[saved] {OUT_DIR}/final", flush=True)


## 8. Training evidence plots

In [ ]:
import os, sys, glob
sys.path.insert(0, os.getcwd())
from pathlib import Path

from kaggle.plot_curves import (
    load_rows, plot_training_loss, plot_reward_curve,
    plot_correctness_rate, plot_before_after,
)
from IPython.display import Image, display

log_path = Path("/kaggle/working/runs/grpo/log.csv")
out_path = Path("/kaggle/working/artifacts/plots")
out_path.mkdir(parents=True, exist_ok=True)

if log_path.exists():
    rows = load_rows(log_path)
    print(f"loaded {len(rows)} log rows")
    plot_training_loss(rows, out_path / "training_loss.png")
    plot_reward_curve(rows, out_path / "reward_curve.png")
    plot_correctness_rate(rows, out_path / "correctness_rate.png")
    print("generated 3 training curve plots")
    for png in sorted(glob.glob(str(out_path / "*.png"))):
        display(Image(filename=png))
else:
    print(f"WARNING: {log_path} not found - run training cell first")

plot_before_after(out_path / "before_after_kernel.png")

## 9. Output

All outputs saved to `/kaggle/working/` and available in the Output tab.

In [ ]:
import subprocess
subprocess.run(["zip", "-r", "/kaggle/working/arm_gym_results.zip",
                "/kaggle/working/runs/", "/kaggle/working/artifacts/"], check=False)
print("arm_gym_results.zip saved to /kaggle/working/")
print("Download from the Output tab on the right.")

## 10. (Optional) Push LoRA adapter to HuggingFace Hub

In [ ]:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# model.push_to_hub("your-username/arm-gym-grpo-lora")
# tokenizer.push_to_hub("your-username/arm-gym-grpo-lora")
print("uncomment above to push adapter to HF Hub")